In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import gzip
import csv
import plotly.express as px

In [2]:
# read in line by line and only keep movies
filtered_rows = []
with gzip.open('title.basics.tsv.gz', 'rt') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        if row['titleType'] == 'movie':
            filtered_rows.append(row)

In [3]:
# convert to data frame
titles_df = pd.DataFrame(filtered_rows)

In [4]:
titles_df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,\N,100,"Documentary,News,Sport"
2,tt0000502,movie,Bohemios,Bohemios,0,1905,\N,100,\N
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,\N,70,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,\N,90,Drama


In [5]:
len(titles_df)

711800

In [6]:
# drop anything that doesn't have a genre
titles_df = titles_df[titles_df['genres'] != "\\N"]

In [7]:
# drop anything that doesn't have a runtime
titles_df = titles_df[titles_df['runtimeMinutes'] != "\\N"]

In [8]:
# Turn runtime into int
titles_df['runtimeMinutes'] = titles_df['runtimeMinutes'].astype(int)

In [9]:
# drop anything that doesn't have a start year 
titles_df = titles_df[titles_df['startYear'] != "\\N"]

In [10]:
len(titles_df)

415018

In [11]:
titles_df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,\N,100,"Documentary,News,Sport"
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,\N,70,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,\N,90,Drama
8,tt0000679,movie,The Fairylogue and Radio-Plays,The Fairylogue and Radio-Plays,0,1908,\N,120,"Adventure,Fantasy"


In [12]:
# Drop any unnecessary columns
titles_df = titles_df.drop(['titleType', 'tconst', 'isAdult', 'originalTitle', 'endYear'], axis=1)

In [13]:
# Split the genres column into a list
titles_df[['genre_1', 'genre_2', 'genre_3']] = titles_df['genres'].str.split(',', expand=True)

In [14]:
titles_df.head()

,primaryTitle,startYear,runtimeMinutes,genres,genre_1,genre_2,genre_3
0,Miss Jerry,1894,45,Romance,Romance,None,None
1,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",Documentary,News,Sport
3,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",Action,Adventure,Biography
4,The Prodigal Son,1907,90,Drama,Drama,None,None
8,The Fairylogue and Radio-Plays,1908,120,"Adventure,Fantasy",Adventure,Fantasy,None


In [15]:
titles_df.isnull().sum()

primaryTitle           0
startYear              0
runtimeMinutes         0
genres                 0
genre_1                0
genre_2           231310
genre_3           330885
dtype: int64

In [16]:
titles_df['startYear'].unique()

array(['1894', '1897', '1906', '1907', '1908', '1909', '1910', '1911',
       '1913', '1912', '1919', '1914', '1916', '1915', '1936', '1917',
       '1925', '1918', '1920', '1921', '1924', '1922', '1923', '1927',
       '1929', '1926', '1993', '1935', '1928', '1942', '1930', '1931',
       '1932', '1937', '1933', '1950', '1938', '1951', '1939', '1934',
       '1946', '1940', '1944', '1947', '1941', '1952', '1957', '1943',
       '1948', '2001', '1945', '1949', '1953', '1954', '1965', '1983',
       '1980', '1973', '1961', '1958', '1964', '1955', '1956', '1977',
       '1962', '1960', '1959', '1967', '1968', '1963', '1971', '1969',
       '1972', '1966', '1976', '1990', '1970', '1979', '1981', '2020',
       '1975', '1978', '1989', '1974', '1986', '1995', '1987', '1985',
       '2018', '1996', '1984', '1992', '2023', '1982', '1988', '1991',
       '2022', '1994', '2008', '2005', '1998', '2002', '1997', '2009',
       '2017', '2000', '2021', '2014', '2004', '2006', '1999', '2019',
      

In [17]:
titles_df['startYear'] = titles_df['startYear'].astype(int)

In [18]:
# Drop anything before the 1900s
titles_df = titles_df[titles_df['startYear'] >= 1900]
# Drop anything after 2025
titles_df = titles_df[titles_df['startYear'] <= 2025]

In [19]:
titles_df['decade'] = (titles_df['startYear'] // 10) * 10

In [20]:
# Also drop this decade because of too much missing data
titles_df = titles_df[titles_df['decade'] != 1900]
titles_df = titles_df[titles_df['decade'] != 1910]

In [21]:
# Change the decades columns to have an s
titles_df['decade'] = titles_df['decade'].astype(str) + 's'

Made a pivot. From here on, I will look at the movies in the 2020s decade. Find the average runtime for all movies in this decade. Then for each genre, find the difference from that runtime, considering that average as the base. 

In [43]:
base_runtime_2000 = titles_df[titles_df['decade'] == '2000s']['runtimeMinutes'].mean()

In [44]:
print(base_runtime_2000)

89.77521436982734


In [45]:
titles_df.head()

,primaryTitle,startYear,runtimeMinutes,genres,genre_1,genre_2,genre_3,decade
461,Dodge City Trail,1936,56,"Drama,Music,Western",Drama,Music,Western,1930s
925,Charley's Aunt,1925,80,Comedy,Comedy,None,None,1920s
3280,Die Brüder Karamasoff,1920,70,Drama,Drama,None,None,1920s
4116,The Corsican Brothers,1920,60,"Action,Drama,Romance",Action,Drama,Romance,1920s
4118,The Courage of Marge O'Doone,1920,70,Drama,Drama,None,None,1920s


In [46]:
grouped_2000 = titles_df[titles_df['decade'] == '2000s'].groupby('genre_1')['runtimeMinutes'].mean()

In [47]:
grouped_2000_df = pd.DataFrame(grouped_2000).reset_index()

In [48]:
grouped_2000_df

,genre_1,runtimeMinutes
0,Action,105.734298
1,Adult,95.081340
2,Adventure,92.611276
3,Animation,79.484461
4,Biography,87.812500
5,Comedy,95.625859
6,Crime,98.682854
7,Documentary,73.467214
8,Drama,98.150095
9,Family,90.419825


In [49]:
grouped_2000_df = grouped_2000_df.drop(1)

In [50]:
grouped_2000_df['diff'] = grouped_2000_df['runtimeMinutes'] - base_runtime_2000

In [55]:
fig = px.bar(
    grouped_2000_df, 
    x='diff', 
    y='genre_1',
    orientation='h',
    color='diff',
    color_continuous_scale=px.colors.diverging.Spectral,
    color_continuous_midpoint=0,
    title='Difference in Average Runtime by Genre (vs. Overall Average from 2000-2010)',
    custom_data=['runtimeMinutes'],
    width = 700
)

fig.update_traces(
    hovertemplate=(
        'Genre: %{y}<br>'
        'Difference: %{x:.2f} min<br>'
        'Avg Runtime: %{customdata[0]:.2f} min'
    )
)

fig.update_layout(
    xaxis_title='Difference from Overall 2000s Average (minutes)',
    yaxis_title='Genre',
    coloraxis_colorbar=dict(title='Difference (min)'),
    margin=dict(l=150),
    height=600,
    width = 900,
    plot_bgcolor='white',
    annotations=[
        dict(
            text="2000s Overall Average Runtime: 90 min",
            xref="paper", yref="paper",
            x=-0.18, y=1.10,
            showarrow=False,
            font=dict(size=12),
            font_color='grey'
        ),
        dict(
            text="Source: IMDB Title Basics Dataset",
            xref="paper", yref="paper",
            x=0.5, y=-0.16,
            showarrow=False,
            font=dict(size=10),
            font_color='grey'
        ),
    ]
)

fig.show()

In [56]:
fig.write_image("30DayChart-9.png")

In [57]:
fig.write_html("30DayChart-9.html")

In [43]:
# Since there are many movies with Null values in the genre_2 and genre_3 columns, just group by genre_1
genre_grouped = titles_df.groupby(['decade', 'genre_1'])['runtimeMinutes'].mean()

In [25]:
# Alternate grouping 
genre_grouped_2 = titles_df.groupby(['decade', 'genre_1']).size()

In [26]:
genre_grouped

decade  genre_1  
1920s   Action        83.962230
        Adventure     83.551622
        Animation     68.666667
        Biography    104.128205
        Comedy        66.756686
                        ...    
2020s   Sport         89.447368
        Talk-Show     92.310345
        Thriller      94.144319
        War           83.057471
        Western       91.742574
Name: runtimeMinutes, Length: 258, dtype: float64

In [27]:
genre_grouped_2.head()

decade  genre_1  
1920s   Action        556
        Adventure     339
        Animation       3
        Biography      39
        Comedy       1533
dtype: int64

In [44]:
# Unstack to make a wide format data frame
genre_pivot = genre_grouped.unstack('genre_1')

In [29]:
genre_pivot_2 = genre_grouped_2.unstack('genre_1')

In [30]:
genre_pivot.head()

genre_1,Action,Adult,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,...,Mystery,News,Reality-TV,Romance,Sci-Fi,Sport,Talk-Show,Thriller,War,Western
decade,,,,,,,,,,,,,,,,,,,,,
1920s,83.962230,NaN,83.551622,68.666667,104.128205,66.756686,67.311798,74.484076,71.930419,53.000000,...,62.649123,64.0,NaN,75.339623,79.800000,60.000000,NaN,57.666667,100.571429,60.057992
1930s,80.563481,NaN,83.673745,82.857143,99.437500,80.065849,73.625390,65.820946,81.749671,76.625000,...,76.353448,NaN,NaN,81.973684,75.875000,40.666667,NaN,73.000000,103.687500,66.131707
1940s,94.680312,NaN,86.620482,79.000000,102.717647,84.848795,80.647500,66.835570,89.297387,82.806452,...,74.432836,NaN,NaN,89.113208,112.800000,72.000000,NaN,78.416667,86.428571,67.628731
1950s,92.623262,NaN,89.438116,83.520833,103.075188,91.223479,85.687045,72.396739,95.076034,82.969697,...,87.169231,NaN,NaN,97.681416,73.880000,53.000000,NaN,88.217391,90.694444,77.900524
1960s,95.259459,76.1875,93.026855,94.171429,109.657895,93.318663,90.260498,75.837273,96.461453,81.000000,...,98.989583,NaN,NaN,94.384279,85.960784,87.600000,NaN,93.991597,95.179775,88.203390


In [45]:
genre_order = genre_grouped.groupby('genre_1').size().sort_values(ascending=False).index

# Reorder the columns in your pivot table
genre_pivot = genre_pivot[genre_order]

In [46]:
# After looking at initial heatmap, drop these columns because they have a lot of missing data
genre_pivot = genre_pivot.drop(['Sport', 'Adult', 'News', 'Reality-TV', 'Talk-Show', 'Game-Show', 'Film-Noir'], axis=1)

In [32]:
genre_pivot.head()

genre_1,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,History,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
decade,,,,,,,,,,,,,,,,,,,,
1920s,83.962230,83.551622,68.666667,104.128205,66.756686,67.311798,74.484076,71.930419,53.000000,80.565217,95.166667,77.500000,96.000000,82.000000,62.649123,75.339623,79.800000,57.666667,100.571429,60.057992
1930s,80.563481,83.673745,82.857143,99.437500,80.065849,73.625390,65.820946,81.749671,76.625000,97.400000,92.100000,70.305556,76.517241,82.332046,76.353448,81.973684,75.875000,73.000000,103.687500,66.131707
1940s,94.680312,86.620482,79.000000,102.717647,84.848795,80.647500,66.835570,89.297387,82.806452,76.812500,84.833333,71.000000,73.093750,85.603053,74.432836,89.113208,112.800000,78.416667,86.428571,67.628731
1950s,92.623262,89.438116,83.520833,103.075188,91.223479,85.687045,72.396739,95.076034,82.969697,107.694444,96.068966,78.237288,85.210526,96.521739,87.169231,97.681416,73.880000,88.217391,90.694444,77.900524
1960s,95.259459,93.026855,94.171429,109.657895,93.318663,90.260498,75.837273,96.461453,81.000000,98.188235,91.833333,85.381381,93.884615,103.113095,98.989583,94.384279,85.960784,93.991597,95.179775,88.203390


In [48]:
fig = px.imshow(
    genre_pivot,
    labels=dict(x="Genre", y="Decade", color="Avg Runtime (min)"),
    aspect="auto",
    text_auto=".1f",
    color_continuous_scale="RdBu"
)

fig.update_layout(
    title="Average Runtime by Decade and Genre",
    xaxis_title="Genre",
    yaxis_title="Decade",
    width = 700,
    height = 500
)

fig.show()

In [34]:
fig = px.imshow(
    genre_pivot_2,
    labels=dict(x="Genre", y="Decade", color="No. of Movies"),
    aspect="auto",
    text_auto=".1f",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    title="No. of movies by Decade and Genre",
    xaxis_title="Genre",
    yaxis_title="Decade"
)

fig.show()